### neural network study

这里的“学习”是指从**训练数据中自动获取最优权重参数的过程**。本章中，为了使神经网络能进行学习，将导入**损失函数这一指标**。而学习的目的就是以该损失函数为基准，找出能使它的值达到最小的权重参数。为了找出尽可能小的损失函数的值，本章我们将介绍利用了函数斜率的梯度法。

#### 从数据中学习
神经网络的特征就是可以从数据中学习。所谓“从数据中学习”，是指可以**由数据自动决定权重参数的值**。这是非常了不起的事情！因为如果所有的参数都需要人工决定的话，工作量就太大了。在第2章介绍的感知机的例子中，我们对照着真值表，人工设定了参数的值，但是那时的参数只有3个。而在实际的神经网络中，**参数的数量成千上万**，在层数更深的深度学习中，参数的数量甚至可以上亿，想要人工决定这些参数的值是不可能的。本章将介绍神经网络的学习，即利用数据决定参数值的方法，并用Python实现对MNIST手写数字数据集的学习。
> 对于线性可分问题，第2章的感知机是可以利用数据自动学习的。根据“感知机收敛定理”，通过有限次数的学习，线性可分问题是可解的。但是，非线性可分问题则无法通过（自动）学习来解决。

##### 数据驱动
机器学习的方法则极力避免人为介入，尝试从收集到的数据中发现答案（模式）。神经网络或深度学习则比以往的机器学习方法更能避免人为介入。
现在我们来思考一个具体的问题，比如如何实现数字“5”的识别。数字5是图4-1所示的手写图像，我们的目标是实现能区别是否是5的程序。这个问题看起来很简单，大家能想到什么样的算法呢？

<img src="data/neural_study/image1.png" style="width:50%;">

如果让我们自己来设计一个能将5正确分类的程序，就会意外地发现这是一个很难的问题。人可以简单地识别出5，但却很难明确说出是基于何种规律而识别出了5。此外，从图4-1中也可以看到，每个人都有不同的写字习惯，要发现其中的规律是一件非常难的工作。<br>
因此，与其绞尽脑汁，从零开始想出一个可以识别5的算法，不如考虑**通过有效利用数据来解决**这个问题。一种方案是，先从图像中**提取特征量**，再用机器学习技术学习这些特征量的模式。这里所说的“特征量”是指可以从输入数据（输入图像）中准确地提取本质数据（重要的数据）的转换器。图像的特征量通常表示为向量的形式。在计算机视觉领域，常用的特征量包括 SIFT、SURF 和 HOG 等。使用这些特征量将图像数据转换为向量，然后对转换后的向量使用机器学习中的SVM、KNN等分类器进行学习。<br>
机器学习的方法中，由机器从收集到的数据中找出规律性。与从零开始想出算法相比，这种方法可以更高效地解决问题，也能减轻人的负担。但是需要注意的是，**将图像转换为向量时使用的特征量仍是由人设计的**。对于不同的问题，必须使用合适的特征量（必须设计专门的特征量），才能得到好的结果。比如，为了区分狗的脸部，人们需要考虑与用于识别5的特征量不同的其他特征量。也就是说，即使使用特征量和机器学习的方法，也需要针对不同的问题人工考虑合适的特征量。<br>
到这里，我们介绍了两种针对机器学习任务的方法。将这两种方法用图来表示，如图4-2所示。图中还展示了神经网络（深度学习）的方法，可以看出该方法不存在人为介入。<br>
如图4-2所示，神经网络直接学习图像本身。在第2个方法，即利用特征量和机器学习的方法中，特征量仍是由人工设计的，而在神经网络中，连 图像中包含的重要特征量也都是由机器来学习的。

<img src="data/neural_study/image2.png" style="width:50%;">

> 深度学习有时也称为端到端机器学习（end-to-end machine learning）。这里所说的端到端是指从一端到另一端的意思，也就是从原始数据（输入）中获得目标结果（输出）的意思。

神经网络的优点是**对所有的问题都可以用同样的流程来解决**。比如，不管要求解的问题是识别5，还是识别狗，抑或是识别人脸，神经网络都是通过不断地学习所提供的数据，尝试发现待求解的问题的模式。也就是说，与待处理的问题无关，**神经网络可以将数据直接作为原始数据，进行“端对端”的学习**。

##### 训练数据和测试数据
本章主要介绍神经网络的学习，不过在这之前，我们先来介绍一下机器 学习中有关数据处理的一些注意事项。<br>
机器学习中，一般将数据分为**训练数据和测试数据**两部分来进行学习和实验等。首先，使用训练数据进行学习，寻找最优的参数；然后，使用测试数据评价训练得到的模型的实际能力。为什么需要将数据分为训练数据和测试数据呢？因为我们追求的是**模型的泛化能力**。为了正确评价模型的泛化能力，就必须划分训练数据和测试数据。另外，训练数据也可以称为监督数据。<br>
泛化能力是指处理未被观察过的数据（不包含在训练数据中的数据）的能力。**获得泛化能力是机器学习的最终目标**。比如，在识别手写数字的问题中，泛化能力可能会被用在自动读取明信片的邮政编码的系统上。此时，手写数字识别就必须具备较高的识别“某个人”写的字的能力。注意这里不是“特定的某个人写的特定的文字”，而是“任意一个人写的任意文字”。如果系统只能正确识别已有的训练数据，那有可能是只学习到了训练数据中的个人的习惯写法。<br>
因此，仅仅用一个数据集去学习和评价参数，是无法进行正确评价的。这样会导致可以顺利地处理某个数据集，但无法处理其他数据集的情况。顺便说一下，只对某个数据集**过度拟合的状态称为过拟合**（over fitting）。避免过拟合也是机器学习的一个重要课题。

#### 损失函数
神经网络的学习通过某个指标表示现在的状态。然后，以这个指标为基准，寻找最优权重参数。<br>
神经网络以某个指标为线索寻找最优权重参数。神经网络的学习中所用的**指标称为损失函数**（loss function）。这个损失函数可以使用任意函数， 但一般用均方误差和交叉熵误差等。
> 损失函数是表示神经网络性能的“恶劣程度”的指标，即当前的神经网络对监督数据在多大程度上不拟合，在多大程度上不一致。以“性能的恶劣程度”为指标可能会使人感到不太自然，但是如果给损失函数乘上一个负值，就可以解释为“在多大程度上不坏”，即“性能有多好”。并且，“使性能的恶劣程度达到最小”和“使性能的优良程度达到最大”是等价的，不管是用“恶劣程度”还是“优良程度”，做的事情本质上都是一样的。

##### 均方误差
可以用作损失函数的函数有很多，其中最有名的是均方误差（mean squared error）。均方误差如下式所示。

$ E = \frac{1}{2} \sum_{k}(y_{k}-t_{k})^{2}\qquad\qquad\qquad\qquad \qquad \rm(4.1) $

这里，$y_k$是表示神经网络的输出，$t_k$表示监督数据，k表示数据的维数。<br>

比如，在3.6节手写数字识别的例子中，$ y_k、t_k $是由如下10个元素构成的数据。<br>
 y = [0.1, 0.05, 0.6, 0.0, 0.05, 0.1, 0.0, 0.1, 0.0, 0.0] <br>
 t = [0, 0, 1, 0, 0, 0, 0, 0, 0, 0] <br>
数组元素的索引从第一个开始依次对应数字“0”“1”“2”……这里，神经网络的输出y是softmax函数的输出。由于softmax函数的输出可以理解为概率，因此上例表示“0”的概率是0.1，“1”的概率是0.05，“2”的概率是0.6等。t是监督数据，将正确解标签设为1，其他均设为0。这里，标签“2”为1，表示正确解是“2”。将正确解标签表示为1，其他标签表示为0的表示方法**称为one-hot表示**。<br>
如式（4.1）所示，均方误差会计算神经网络的输出和正确解监督数据的各个元素之差的平方，再求总和。现在，我们用Python来实现这个均方误差，实现方式如下所示。

In [ ]:
import numpy as np
def mean_squared_error(y, t):
    return 0.5 * np.sum((y - t)**2)

# 设“2”为正确解 
t = [0, 0, 1, 0, 0, 0, 0, 0, 0, 0]
# 例1：“2”的概率最高的情况（0.6） 
y = [0.1, 0.05, 0.6, 0.0, 0.05, 0.1, 0.0, 0.1, 0.0, 0.0]
print(mean_squared_error(np.array(y), np.array(t))) # 0.097500000000000031 
# 例2：“7”的概率最高的情况（0.6）
y = [0.1, 0.05, 0.1, 0.0, 0.05, 0.1, 0.0, 0.6, 0.0, 0.0]    
print(mean_squared_error(np.array(y), np.array(t))) # 0.59750000000000003

0.09750000000000003
0.5975


这里举了两个例子。第一个例子中，正确解是“2”，神经网络的输出的最大值是“2”；第二个例子中，正确解是“2”，神经网络的输出的最大值是“7”。如实验结果所示，我们发现第一个例子的损失函数的值更小，和监督数据之间的误差较小。也就是说，均方误差显示第一个例子的输出结果与监督数据更加吻合。

##### 交叉熵误差
除了均方误差之外，交叉熵误差（cross entropy error）也经常被用作损失函数。交叉熵误差如下式所示。

$ E = -\sum_{k}t_{k}\log y_{k}\qquad\qquad\qquad\qquad\qquad (4.2) $ 

这里，log表示以e为底数的自然对数（$\log \tiny{\mathrm{e}}$）。$y_k$是神经网络的输出，$t_k$ 是正确解标签。并且，$t_k$中只有正确解标签的索引为1，其他均为0（one-hot表示）。 因此，式（4.2）实际上只计算对应正确解标签的输出的自然对数。比如，假设正确解标签的索引是“2”，与之对应的神经网络的输出是0.6，则交叉熵误差是 $ −log_{0.6} = 0.51 $；若“2”对应的输出是0.1，则交叉熵误差为 $ −log_{0.1} = 2.30 $。也就是说，交叉熵误差的值是**由正确解标签所对应的输出结果决定的**。<br>

自然对数的图像如图4-3所示。<br>
<img src="data/neural_study/image3.png" style="width:50%;">

如图4-3所示，x等于1时，y为0；随着x向0靠近，y逐渐变小。因此，正确解标签对应的输出越大，式（4.2）的值越接近0；当输出为1时，交叉熵误差为0。此外，如果正确解标签对应的输出较小，则式（4.2）的值较大。<br>

下面，我们来用代码实现交叉熵误差。

In [4]:
def cross_entropy_error(y, t):
    delta = 1e-7
    return -np.sum(t * np.log(y + delta))

t = [0, 0, 1, 0, 0, 0, 0, 0, 0, 0]
y = [0.1, 0.05, 0.6, 0.0, 0.05, 0.1, 0.0, 0.1, 0.0, 0.0]
print(cross_entropy_error(np.array(y), np.array(t))) # 0.51082545709933802

y = [0.1, 0.05, 0.1, 0.0, 0.05, 0.1, 0.0, 0.6, 0.0, 0.0]
print(cross_entropy_error(np.array(y), np.array(t))) # 2.3025840929945458

0.510825457099338
2.302584092994546


这里，参数y和t是NumPy数组。函数内部在计算np.log时，加上了一个微小值delta。这是因为，当出现np.log(0)时，np.log(0)会变为负无限大的-inf，这样一来就会导致后续计算无法进行。作为保护性对策，**添加一个微小值可以防止负无限大的发生**。下面，我们使用cross_entropy_error(y, t)进行一些简单的计算。<br>
第一个例子中，正确解标签对应的输出为0.6，此时的交叉熵误差大约为0.51。第二个例子中，正确解标签对应的输出为0.1的低值，此时的交叉熵误差大约为2.3。由此可以看出，这些结果与我们前面讨论的内容是一致的。 